# Day 5 — Model Evaluation: Base vs Fine-Tuned

Compare the base GPT-2 against the fine-tuned model from Day 4 using qualitative response comparison and quantitative perplexity metrics.

In [ ]:
import json
import os
import math
import torch
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

---
## Part 1: Load Both Models

In [ ]:
print("Loading base GPT-2...")
base_model = AutoModelForCausalLM.from_pretrained("gpt2")
base_tokenizer = AutoTokenizer.from_pretrained("gpt2")
base_tokenizer.pad_token = base_tokenizer.eos_token

ft_path = "../day-04-supervised-fine-tuning/gpt2-cybersecurity-sft"
if not os.path.exists(ft_path):
    print(f"Fine-tuned model not found at {ft_path}. Run Day 4 first.")
    exit()

print("Loading fine-tuned model...")
ft_model = AutoModelForCausalLM.from_pretrained(ft_path)
ft_tokenizer = AutoTokenizer.from_pretrained(ft_path)
ft_tokenizer.pad_token = ft_tokenizer.eos_token
print("Both models loaded.")

---
## Part 2: Side-by-Side Response Comparison

Test both models on the same prompts and compare outputs.

In [ ]:
test_prompts = [
    "What is a firewall?",
    "Explain how encryption works.",
    "What is a DDoS attack?",
    "Define social engineering.",
    "What is ransomware?",
]

base_pipe = pipeline("text-generation", model=base_model, tokenizer=base_tokenizer)
ft_pipe = pipeline("text-generation", model=ft_model, tokenizer=ft_tokenizer)

results = []
for prompt in test_prompts:
    fp = f"Instruction:\n{prompt}\n\nResponse:\n"
    
    base_out = base_pipe(fp, max_new_tokens=50, temperature=0.7, do_sample=True, pad_token_id=base_tokenizer.eos_token_id)
    ft_out = ft_pipe(fp, max_new_tokens=50, temperature=0.7, do_sample=True, pad_token_id=ft_tokenizer.eos_token_id)
    
    results.append({
        "prompt": prompt,
        "base": base_out[0]["generated_text"][len(fp):].strip(),
        "ft": ft_out[0]["generated_text"][len(fp):].strip()
    })

for r in results:
    print(f"Prompt: {r['prompt']}")
    print(f"  BEFORE: {r['base'][:100]}")
    print(f"  AFTER:  {r['ft'][:100]}")
    print()

---
## Part 3: Perplexity Comparison

Perplexity measures how well the model predicts text. Lower is better.

In [ ]:
def compute_perplexity(model, tokenizer, texts):
    model.eval()
    total_loss = 0
    total_tokens = 0
    with torch.no_grad():
        for text in texts:
            formatted = f"Instruction:\n{text['instruction']}\n\nResponse:\n{text['response']}"
            encoded = tokenizer(formatted, return_tensors="pt", truncation=True, max_length=256)
            labels = encoded["input_ids"].clone()
            prefix = formatted[:formatted.find("Response:\n") + len("Response:\n")]
            prefix_len = len(tokenizer.encode(prefix))
            labels[:, :prefix_len] = -100
            outputs = model(encoded["input_ids"], labels=labels)
            total_loss += outputs.loss.item() * (labels != -100).sum().item()
            total_tokens += (labels != -100).sum().item()
    return math.exp(total_loss / total_tokens) if total_tokens > 0 else 0

val_path = "../day-03-dataset-preparation/cybersecurity_val.jsonl"
if os.path.exists(val_path):
    with open(val_path) as f:
        val_data = [json.loads(line) for line in f]
else:
    val_data = [{"instruction": "What is a firewall?", "response": "A firewall is a security device that monitors traffic."}]

base_ppl = compute_perplexity(base_model, base_tokenizer, val_data)
ft_ppl = compute_perplexity(ft_model, ft_tokenizer, val_data)

print(f"Base model perplexity:       {base_ppl:.2f}")
print(f"Fine-tuned model perplexity: {ft_ppl:.2f}")
print(f"Improvement:                 {((base_ppl - ft_ppl) / base_ppl * 100):.1f}%")

---
## Visualization

In [ ]:
plt.figure(figsize=(14, 5))

# Perplexity
plt.subplot(1, 2, 1)
bars = plt.bar(["Base GPT-2", "Fine-Tuned"], [base_ppl, ft_ppl], color=['lightcoral', 'steelblue'], edgecolor='navy')
plt.title("Perplexity Comparison (Lower is Better)", fontsize=13)
plt.ylabel("Perplexity")
plt.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, [base_ppl, ft_ppl]):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f"{val:.1f}", ha='center', fontsize=12)

# Response length
plt.subplot(1, 2, 2)
avg_base = sum(len(r["base"]) for r in results) / len(results)
avg_ft = sum(len(r["ft"]) for r in results) / len(results)
bars = plt.bar(["Base GPT-2", "Fine-Tuned"], [avg_base, avg_ft], color=['lightcoral', 'steelblue'], edgecolor='navy')
plt.title("Average Response Length (Characters)", fontsize=13)
plt.ylabel("Characters")
plt.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, [avg_base, avg_ft]):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f"{val:.0f}", ha='center', fontsize=12)

plt.tight_layout()
plt.show()

## Key Takeaways

- **Perplexity** quantifies how well a model predicts text (lower = better)
- The fine-tuned model should show lower perplexity on the validation set
- **Qualitative comparison** is equally important — read the actual responses
- Base models generate generic completions; fine-tuned models produce domain-specific answers
- Always evaluate on data the model hasn't seen during training
- Both quantitative metrics and human judgment are needed for thorough evaluation